# Sinling POS Tagging

This notebook loads the `century_tokens.json` file and passes each sentence into the [Sinling POS Tagger](https://github.com/ysenarath/sinling) to generate Part-Of-Speech tags for every word. The final output is saved to `data/sinling_pos_tags.csv`.

In [1]:
!pip install sinling


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import os
import pandas as pd
from tqdm import tqdm
from sinling import POSTagger

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
# Configuration
INPUT_FILE = '../data/century_tokens.json'
OUTPUT_CSV = '../data/sinling_pos_tags.csv'

# Load Data
if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        century_data = json.load(f)
    print(f"Loaded tokens from {INPUT_FILE}")
else:
    print(f"File not found: {INPUT_FILE}")
    century_data = {}

Loaded tokens from ../data/century_tokens.json


In [5]:
# Initialize Sinling POS Tagger
tagger = POSTagger()

rows = []

# Loop through each century and sentence
for century_key, sentences in tqdm(century_data.items(), desc="Processing Centuries"):
    for sentence_idx, tokens in enumerate(sentences):
        if not tokens:
            continue
            
        # The tagger expects a list of tokenized sentences, e.g. [[word1, word2, ...]]
        # It returns a list of tagged sentences: [[(word1, tag1), (word2, tag2), ...]]
        try:
            tagged_sentence = tagger.predict([tokens])[0]
            
            # Extract tags and append to DataFrame rows
            for word, tag in tagged_sentence:
                rows.append({
                    'century': century_key,
                    'sentence_index': sentence_idx,
                    'word': word,
                    'sinling_pos': tag
                })
        except Exception as e:
            print(f"Error processing century {century_key}, sentence {sentence_idx}: {e}")
            continue

Processing Centuries:   0%|          | 0/9 [00:00<?, ?it/s]

Processing Centuries: 100%|██████████| 9/9 [00:00<00:00, 20.17it/s]


In [6]:
# Create DataFrame
df_pos = pd.DataFrame(rows)
print(f'Total words tagged: {len(df_pos)}')

# Save as CSV
df_pos.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"Successfully saved tags to {OUTPUT_CSV}")

df_pos.head(20)

Total words tagged: 64813
Successfully saved tags to ../data/sinling_pos_tags.csv


,century,sentence_index,word,sinling_pos
0,5,0,කාම,UNK
1,5,0,ලොක,UNK
2,5,0,රූප,UNK
3,5,0,ලොක,UNK
4,5,0,අරූපලෝක,UNK
5,5,0,යන්,NNC
6,5,0,ලොකත්රයට,NNC
7,5,0,ශ්රෙෂ්ඨ,JCV
8,5,0,වූ,VP
9,5,0,මුනීන්ද්ර,NNP


In [7]:
df_pos['sinling_pos'].value_counts()

sinling_pos
NNC      24122
JJ        7229
VP        6978
NNP       4373
POST      4291
VNF       3179
DET       1910
PRP       1652
RP        1641
FS        1378
NNJ        928
JCV        892
VNN        837
CC         741
NUM        677
NIP        658
NCV        648
UNK        625
RB         432
RRPCV      357
VFM        321
AUX        280
CM         227
ABB        173
PUNC       100
QBE         85
NVB         48
QUE         24
NDT          7
Name: count, dtype: int64

In [8]:
df_pos[df_pos['sinling_pos'] == 'RRPCV'].head(20)

,century,sentence_index,word,sinling_pos
130,5,8,පට,RRPCV
350,5,20,මිය,RRPCV
467,5,26,වහමු,RRPCV
576,12,1,මුත්,RRPCV
691,12,12,පවත්,RRPCV
699,12,13,මැඩ,RRPCV
779,12,17,පවත්,RRPCV
847,12,19,පිළි,RRPCV
1052,12,36,රැස්,RRPCV
1062,12,37,දැන,RRPCV


In [9]:
list(df_pos['sinling_pos'].value_counts().to_dict().keys())

['NNC',
 'JJ',
 'VP',
 'NNP',
 'POST',
 'VNF',
 'DET',
 'PRP',
 'RP',
 'FS',
 'NNJ',
 'JCV',
 'VNN',
 'CC',
 'NUM',
 'NIP',
 'NCV',
 'UNK',
 'RB',
 'RRPCV',
 'VFM',
 'AUX',
 'CM',
 'ABB',
 'PUNC',
 'QBE',
 'NVB',
 'QUE',
 'NDT']

In [10]:
sinmorphy_df = pd.read_csv('../data/century_morphology_data_v6.csv')

In [11]:
sinmorphy_df.head()

,century,sentence_index,word,new_lemma,morphology,pos,all_tags,all_lemmas,all_morphologies
0,5,0,කාම,කාම,කාම+N+NUTR+PL+NOM,NOUN,NOUN,කාම,කාම+N+NUTR+PL+NOM
1,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC
2,5,0,රූප,රූප,රූප+N+NUTR+PL+NOM,NOUN,NOUN,රූප,රූප+N+NUTR+PL+NOM
3,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC
4,5,0,අරූපලෝක,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC


In [12]:
sinmorphy_df['pos'].value_counts()

pos
NOUN                      37095
Proper noun or Invalid     9143
Verb                       8644
PART                       5788
PROPER NOUN                2264
ADJECTIVE                  1316
PRONOUN                     546
FOREIGN WORD                 17
Name: count, dtype: int64

In [13]:
list(sinmorphy_df['pos'].value_counts().to_dict().keys())

['NOUN',
 'Proper noun or Invalid',
 'Verb',
 'PART',
 'PROPER NOUN',
 'ADJECTIVE',
 'PRONOUN',
 'FOREIGN WORD']

In [14]:
pos_map = {
    'NOUN': [
        'NNC',  # Common Nouns [cite: 195]
        'NNJ',  # Adjectival Noun [cite: 90]
        'NUM',  # Number/Numeral
        'NVB'   # Nominalized Verb
    ],
    'PROPER NOUN': [
        'NNP'   # Proper Noun [cite: 203]
    ],
    'PRONOUN': [
        'PRP',  # Pronoun [cite: 199]
        'QUE',  # Questioning Pronouns [cite: 209]
        'NDT',  # Deterministic Pronoun [cite: 212]
        'QBE'   # Question Based Pronoun [cite: 216]
    ],
    'Verb': [
        'VFM',  # Verb Finite [cite: 221]
        'VNF',  # Verb Non-Finite
        'VNN',  # Verbal Noun
        'VP',   # Verb Participle [cite: 44]
        'AUX',  # Auxiliary Verb
        'NCV',  # Noun in Compound Verb [cite: 64]
        'JCV',  # Adjective in Compound Verb (also referred to as ACV) [cite: 67]
        'RRPCV' # Relative Participle in Compound Verb
    ],
    'ADJECTIVE': [
        'JJ'    # Adjective [cite: 2]
    ],
    'PART': [
        'NIP',  # Nipāta (General Particle) [cite: 2]
        'POST', # Postposition [cite: 120]
        'DET',  # Determiner [cite: 122]
        'RP',   # Particle
        'CC',   # Conjunction
        'CM'    # Case Marker
    ],
    'FOREIGN WORD': [
        'ABB',  # Abbreviation [cite: 155]
        'RB',   # Adverb (Level 1 category) [cite: 2]
        'FS',   # Full Stop [cite: 155]
        'PUNC'  # Punctuation [cite: 155]
    ],
    'UNKNOWN': [
        'UNK'   # Unknown word
    ]
}

In [15]:
df_pos["sinling_upd_pos"] = df_pos['sinling_pos'].map({tag: k for k, v in pos_map.items() for tag in v})

In [16]:
df_pos.head(20)

,century,sentence_index,word,sinling_pos,sinling_upd_pos
0,5,0,කාම,UNK,UNKNOWN
1,5,0,ලොක,UNK,UNKNOWN
2,5,0,රූප,UNK,UNKNOWN
3,5,0,ලොක,UNK,UNKNOWN
4,5,0,අරූපලෝක,UNK,UNKNOWN
5,5,0,යන්,NNC,NOUN
6,5,0,ලොකත්රයට,NNC,NOUN
7,5,0,ශ්රෙෂ්ඨ,JCV,Verb
8,5,0,වූ,VP,Verb
9,5,0,මුනීන්ද්ර,NNP,PROPER NOUN


In [17]:
df_pos.to_csv('../data/century_morphology_data_sinling_full.csv', index=False, encoding='utf-8-sig')